# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmad-Fareed/ML_Internship_Flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Ahmad-Fareed/ML_Internship_Flyrank"
REPO_DIR = "ML_Internship_Flyrank"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{df.shape[0]:,} rows x {df.shape[1]} columns loaded.")

30,000 rows x 44 columns loaded.


## 1. My lane (or freestyle) and why

**Lane 3 — Structured Content Archetype Clustering**

I am choosing Lane 3 because the starter data has 30,000 content items spanning 3 content types and 32 clients, with wildly different performance profiles. Some pages get 500,000+ impressions while others get single digits; some are 90 days old, others over 500 days. A single "refresh" or "don't refresh" decision treats all these pages the same — but they are clearly not the same.

Clustering will find **groups of pages that behave alike** (archetypes), and each archetype gets its own recommended action: protect, improve, rewrite, monitor, or prune. This is more useful than a one-size-fits-all ranked list because an editor can apply different playbooks to different archetype groups.

The question:
> **What distinct performance archetypes exist across the content inventory, and what action does each archetype suggest?**

In [2]:
# Quick evidence: content types exist, and they behave very differently
print("=== Content type distribution ===")
print(df["content_type"].value_counts())
print()
print("=== Trend direction distribution ===")
print(df["trend_direction"].value_counts())

=== Content type distribution ===
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

=== Trend direction distribution ===
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. The question: decision, action, cost of a wrong call

**Decision this improves:** Which pages get which treatment. Instead of ranking all 30,000 pages in one queue, an editor sees archetype groups — "these 2,000 pages are high-visibility stable champions (protect them), these 5,000 are declining with demand (refresh them first), these 3,000 are low-demand stale pages (consider pruning or merging)."

**Who acts:** A FlyRank content editor or strategist. They look at the archetype profile (typical impressions, CTR, age, trend) and apply the matching playbook.

**What a wrong recommendation costs:**
- Labeling a healthy page as "prune" wastes content that is quietly performing.
- Labeling a declining page as "protect" wastes time — it needed attention.
- The cost is moderate: the worst case is wasted editor hours, not lost revenue, because a human still reviews before acting.

**Why data/ML helps:** The content inventory is too large (30,000+ pages, growing with the warehouse) and too multi-dimensional (impressions, position, CTR, engagement, age, word count, trend) for a human to eyeball. A clustering algorithm can find natural groupings across all these dimensions simultaneously — groupings that a hand-written rule might miss because the signals are tangled.

In [3]:
# Evidence: pages vary wildly across key metrics — too much for a human to sort by hand
metrics = ["impressions_90d", "sessions_90d", "avg_position", "ctr",
           "engagement_rate", "word_count", "content_age_days"]
print("=== Key metric spreads (showing pages behave very differently) ===")
print(df[metrics].describe().round(1).to_string())

=== Key metric spreads (showing pages behave very differently) ===
       impressions_90d  sessions_90d  avg_position      ctr  engagement_rate  word_count  content_age_days
count          30000.0       30000.0       30000.0  30000.0          30000.0     22301.0           30000.0
mean            5200.4          37.1          16.3      0.5              2.5      3107.8             256.2
std            16838.0         107.1          15.2      3.3              8.3      1452.4             132.7
min                1.0           1.0           0.0      0.0              0.0         8.0              90.0
25%               81.0           2.0           6.2      0.0              0.0      2413.0             132.0
50%              731.0           7.0          10.8      0.1              0.0      2877.0             236.0
75%             3615.2          27.0          22.3      0.3              1.4      3666.0             333.0
max           517715.0        4345.0         245.0    100.0            100.0 

## 3. Quick look at the data (2-3 real numbers)

Three numbers that make this lane worth 7 weeks:

In [4]:
# NUMBER 1: Content types already have drastically different metric profiles
print("=== Number 1: Median metrics by content type ===")
grouped = df.groupby("content_type")[
    ["impressions_90d", "sessions_90d", "ctr", "engagement_rate",
     "word_count", "content_age_days", "avg_position"]
].median()
print(grouped.round(1).to_string())
print()
print("Observation: keyword articles get 955 median impressions vs feedly articles at 4.")
print("These are fundamentally different page types — clustering should find sub-groups within each.")

=== Number 1: Median metrics by content type ===
                    impressions_90d  sessions_90d  ctr  engagement_rate  word_count  content_age_days  avg_position
content_type                                                                                                       
comparison article            107.0           3.0  0.0              0.0      3155.0             151.0           8.5
feedly article                  4.0           3.0  0.0              0.0      1237.0             291.0           4.0
keyword article               955.0           8.0  0.1              0.0      2902.0             236.0          11.7

Observation: keyword articles get 955 median impressions vs feedly articles at 4.
These are fundamentally different page types — clustering should find sub-groups within each.


In [5]:
# NUMBER 2: Declining rates differ by content type — archetypes exist within trend behavior
print("=== Number 2: Declining rate by content type ===")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
print(df.groupby("content_type")["is_declining"].mean().round(3))
print()
print("Observation: 57.2% of comparison articles are declining vs only 28.7% of feedly articles.")
print("Content type alone creates different behavioral profiles — clustering can find finer groups.")

=== Number 2: Declining rate by content type ===
content_type
comparison article    0.572
feedly article        0.287
keyword article       0.561
Name: is_declining, dtype: float64

Observation: 57.2% of comparison articles are declining vs only 28.7% of feedly articles.
Content type alone creates different behavioral profiles — clustering can find finer groups.


In [6]:
# NUMBER 3: Impressions span 5 orders of magnitude — there are natural tiers
print("=== Number 3: Impression spread ===")
percentiles = df["impressions_90d"].quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.99])
print(percentiles.round(0).to_string())
print()
print(f"Range: {df['impressions_90d'].min():.0f} to {df['impressions_90d'].max():,.0f}")
print("A page with 517,715 impressions and a page with 1 impression are not the same archetype.")
print("Clustering will separate these naturally, so each group gets the right action.")

=== Number 3: Impression spread ===
0.10        5.0
0.25       81.0
0.50      731.0
0.75     3615.0
0.90    12136.0
0.99    73506.0

Range: 1 to 517,715
A page with 517,715 impressions and a page with 1 impression are not the same archetype.
Clustering will separate these naturally, so each group gets the right action.


## 4. Careful words: what I can and can't claim

**What I CAN claim (observed / directional / decision-support):**
- I can identify groups of pages that share similar measurable performance characteristics (impressions, position, CTR, age, engagement, trend direction).
- I can describe each group's typical profile in plain numbers and suggest which action category fits best (protect, refresh, prune, monitor, expand).
- I can say: "Pages in this cluster are *observed* to be older, higher-traffic, and declining — the directional recommendation is to review them for refresh first."
- This is **decision-support**: it helps an editor spend their limited time on the most promising group, not a guarantee of outcome.

**What I CANNOT claim:**
- I cannot claim these archetypes are "real" categories in any absolute sense — clustering is a lens, not a truth.
- I cannot claim that acting on the recommendation (e.g., refreshing a declining page) will *cause* recovery. That would require a controlled experiment.
- I cannot claim to have "predicted Google's algorithm" or proven any causal link between content attributes and search ranking.
- I cannot claim the clusters found on the 30k starter slice will hold identically on the full 79M-row warehouse — that needs separate validation.

In [10]:
# Final summary: why this is NOT just "train a model"
print("Why this is not just 'train a model':")
print("- Clustering is unsupervised — there is no label to predict.")
print("- The value is in the PROFILES and the ACTION MAPPING, not in a score.")
print("- A ranked list says review this page first archetypes say")
print("  'this GROUP of pages needs THIS kind of attention'.")
print("- The deliverable is a human-readable archetype report with")
print("  recommended actions, not just a number.")

Why this is not just 'train a model':
- Clustering is unsupervised — there is no label to predict.
- The value is in the PROFILES and the ACTION MAPPING, not in a score.
- A ranked list says review this page first archetypes say
  'this GROUP of pages needs THIS kind of attention'.
- The deliverable is a human-readable archetype report with
  recommended actions, not just a number.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.